# 🥇 Vesuvius Bulletproof Pipeline — Proprietary Edition
**7.5-hour runtime · Kaggle-safe · OOM-resistant · Topology-optimized · Zero external model imports**

> **§0 Non-negotiable ground rules:**
> - Submitting a **system**, not a model — 100% proprietary architectures built from raw `torch.nn`
> - Single file: `/kaggle/working/submission.zip`
> - Inside ZIP: exactly one `.tif` per ID in `test.csv` — filename = `{id}.tif`, no folders, no extras
> - Every `.tif`: matches test volume shape, `dtype uint8`, values `{0, 1}` only
> - **Never skip a test volume**, even if OOM occurs. Degrade compute, never skip.

In [ ]:
# ============================================================
# §0  TIME TRACKING  &  ENVIRONMENT BOOTSTRAP
# ============================================================
import time, os, sys, gc, warnings, math, random
warnings.filterwarnings("ignore")

pipeline_start = time.time()
MAX_RUNTIME   = 9 * 3600        # Kaggle hard-kill
SAFETY_BUFFER = 30 * 60         # 30 min reserve
INF_DEADLINE  = 7.5 * 3600      # Must start inference by 7.5 h

def elapsed_h():
    return (time.time() - pipeline_start) / 3600

def remaining_h():
    return (MAX_RUNTIME / 3600) - elapsed_h()

def check_budget(tag=""):
    e, r = elapsed_h(), remaining_h()
    print(f"[TIME] {tag}: {e:.2f} h elapsed, {r:.2f} h left")
    if r < SAFETY_BUFFER / 3600:
        raise RuntimeError(f"[FATAL] Only {r:.2f} h left — safety buffer breached")
    return e, r

check_budget("Pipeline start")

# tifffile + imagecodecs (needed for LZW-compressed training TIFs)
# Internet is OFF on submission — must use pre-installed packages only.
try:
    import tifffile
except ImportError:
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile"])
    except Exception:
        raise RuntimeError("[FATAL] tifffile not available and pip install failed (internet off?)")
    import tifffile

HAS_IMAGECODECS = False
try:
    import imagecodecs  # noqa: F401 — required by tifffile for LZW/ZSTD decoding
    HAS_IMAGECODECS = True
    print("[OK] imagecodecs available")
except ImportError:
    print("[INFO] imagecodecs not installed — will use Pillow fallback for compressed TIFs")

from PIL import Image
import numpy as np  # needed early for read_tif_volume

def read_tif_volume(path):
    """
    Read a 3D TIF volume. Uses tifffile if imagecodecs is available,
    otherwise falls back to Pillow (handles LZW without imagecodecs).
    """
    if HAS_IMAGECODECS:
        return tifffile.imread(path)
    # Pillow fallback — read multi-page TIF frame by frame
    img = Image.open(path)
    frames = []
    for i in range(img.n_frames):
        img.seek(i)
        frames.append(np.array(img))
    img.close()
    return np.stack(frames, axis=0)

print("[OK] Environment ready")

In [ ]:
# ============================================================
# §0  IMPORTS  &  CONFIGURATION CONSTANTS
# ============================================================
import numpy as np
import pandas as pd
import zipfile
import scipy.ndimage as ndi
from pathlib import Path
from skimage.morphology import remove_small_objects
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] PyTorch {torch.__version__}  device={DEVICE}")
if DEVICE.type == "cuda":
    print(f"     GPU: {torch.cuda.get_device_name(0)}  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Reproducibility ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Paths ──
ROOT_DIR       = "/kaggle/input/vesuvius-challenge-surface-detection"
TRAIN_IMG_DIR  = f"{ROOT_DIR}/train_images"       # training volumes
TRAIN_LBL_DIR  = f"{ROOT_DIR}/train_labels"       # training labels
TEST_DIR       = f"{ROOT_DIR}/test_images"
OUTPUT_DIR     = "/kaggle/working/submission_masks"
ZIP_PATH       = "/kaggle/working/submission.zip"
CKPT_DIR       = "/kaggle/working/checkpoints"
PRETRAINED_DIR = "/kaggle/input/vesuvius-trained-weights-new"  # attach .pt files as Kaggle dataset
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Model configuration ──
NUM_CLASSES    = 2                        # background + surface
INPUT_SHAPE    = (128, 128, 128)          # (D, H, W) training patch
BASE_CHANNELS  = 32                       # first conv width
MODEL_C_BASE   = 40                       # Model C: wider channels for Z-specialist
MODEL_C_PATCH  = (160, 96, 96)            # Model C: Z-heavy training patch (D, H, W)

# ── Ensemble (§4) — 5 diverse architectures ──
ENSEMBLE_MODE    = True
ENSEMBLE_METHOD  = "mean"                  # "mean" or "median"
ENSEMBLE_WEIGHTS = [0.40, 0.35, 0.25]               # [D, C, A] — 3-model ensemble
# 3-model ensemble: only D, C, A (dropped B and E for more training epochs)

# ── Training (§2 budget) ──
# Auto-detect: skip training if pretrained weights exist (3 models: D, C, A)
_pretrained_found = all(
    os.path.exists(f"/kaggle/input/vesuvius-trained-weights-new/{t}.pt")
    for t in ["model_d", "model_c", "model_a"]
)
TRAIN_ENABLED     = not _pretrained_found  # False when pretrained .pt files are attached
if _pretrained_found:
    print("[OK] Pretrained weights detected → skipping training (inference only)")
MAX_TRAIN_HOURS   = 6.0                   # hard cap (2h per model × 3, only used if training)
TRAIN_EPOCHS      = 7                     # budget allows ~4-5/model; T_max=7 → cosine ~70% complete
TRAIN_LR          = 1e-3
TRAIN_BATCH       = 2
TRAIN_PATCH       = INPUT_SHAPE
FG_OVERSAMPLE     = 0.70                  # fraction of patches with foreground
EMA_DECAY         = 0.999                 # exponential moving average for weights
MAX_TRAIN_VOLS    = 40                    # match offline training config

# ── Inference (§5-7) ──
TTA_ENABLED    = True                    # basic TTA with top-3 models fits 120-vol budget

# ── Postprocessing (§8) — tuned for competition metric ──
# Score = 0.30×TopoScore + 0.35×SurfaceDice@τ=2 + 0.35×VOI_score
# Strategy: avoid cross-wrap bridges (kills VOI+Topo), preserve within-wrap continuity
PP_T_LOW       = 0.35                    # raised from 0.30: less propagation → fewer bridges
PP_T_HIGH      = 0.65                    # lowered from 0.80: capture more surface for SurfaceDice
PP_Z_RADIUS    = 1                       # reduced from 3: avoid bridging adjacent wraps
PP_XY_RADIUS   = 1                       # reduced from 2: avoid bridging adjacent wraps
PP_DUST_MIN    = 500                     # raised from 100: cleaner topology for TopoScore/VOI
PP_BOUNDARY_W  = 0.5                     # boundary loss weight (targets SurfaceDice@τ)
PP_SMOOTH_SIGMA = 0.5                    # Gaussian smoothing on prob map before threshold

# ── Debug (§9) ──
DEBUG_VIZ      = False                    # MUST be False for submission

# ── Per-volume time budget ──
MAX_SEC_PER_VOL = 240                    # ~120 vols × 240s = 8h (within 9h budget)


print("[OK] All configuration loaded")
check_budget("Config done")

## §1 Dataset & ID Contract / §2 Runtime Budget

| Rule | Detail |
|------|--------|
| **Use only** | `test.csv` + `test_images/` for inference; `train_images/` + `train_labels/` for training |
| **Ignore** | `deprecated_train_images`, `deprecated_train_labels` |
| **IDs** | From `test.csv` only — never directory listing, never hard-coded |
| **Missing test.csv** | Fail early |
| **Failed inference** | Degrade compute, **never skip** |
| **Budget** | Training ≤5 h · Inference ≤2 h · Postprocess+ZIP ~15 min · Safety ≥30 min |

In [ ]:
# ============================================================
# §1  DATASET & ID CONTRACT  /  §2  RUNTIME BUDGET
# ============================================================

# ── Hard gate: test.csv ──
test_csv_path = f"{ROOT_DIR}/test.csv"
assert os.path.exists(test_csv_path), f"[FATAL] test.csv not found at {test_csv_path}"

test_df   = pd.read_csv(test_csv_path)
test_ids  = list(dict.fromkeys(test_df["id"].astype(str).tolist()))  # deduplicate, preserve order
print(f"[OK] test.csv: {len(test_ids)} unique test volumes  (sample: {test_ids[:5]})")

# Validate test volumes exist
missing = [v for v in test_ids if not os.path.exists(f"{TEST_DIR}/{v}.tif")]
if missing:
    print(f"[WARN] {len(missing)} missing test volumes: {missing[:10]}")
else:
    print(f"[OK] All {len(test_ids)} test volumes found on disk")

# ── Discover training data (NEVER scan deprecated_* dirs) ──
train_vol_ids = []
for candidate_dir in [TRAIN_IMG_DIR]:
    if os.path.isdir(candidate_dir):
        for f in sorted(os.listdir(candidate_dir)):
            if f.endswith(".tif"):
                vid = f.replace(".tif", "")
                lbl_path = os.path.join(
                    candidate_dir.replace("images", "labels"), f
                )
                if os.path.exists(lbl_path):
                    train_vol_ids.append({
                        "id": vid,
                        "image": os.path.join(candidate_dir, f),
                        "label": lbl_path,
                    })

_total_train_found = len(train_vol_ids)
if len(train_vol_ids) > MAX_TRAIN_VOLS:
    random.shuffle(train_vol_ids)
    train_vol_ids = train_vol_ids[:MAX_TRAIN_VOLS]
    print(f"[OK] Subsampled to {MAX_TRAIN_VOLS} training volumes (from {_total_train_found} found)")
else:
    print(f"[OK] Found {len(train_vol_ids)} training volumes with labels")
if len(train_vol_ids) == 0:
    TRAIN_ENABLED = False
    print("[WARN] No training data \u2014 will attempt to load pre-trained weights")

# ── Budget ──
n_test = len(test_ids)
est_inf_h = n_test * MAX_SEC_PER_VOL / 3600

# Dynamic training cap: 9h total - inference - 1h (safety+overhead)
MAX_TRAIN_HOURS = max(1.0, min(MAX_TRAIN_HOURS, (MAX_RUNTIME / 3600) - est_inf_h - 1.0))


print(f"\n[BUDGET] {n_test} vols \u00d7 {MAX_SEC_PER_VOL}s = {est_inf_h:.1f} h inference est.")
print(f"[BUDGET] Dynamic training cap: {MAX_TRAIN_HOURS:.1f} h")
print(f"[BUDGET] Remaining: {remaining_h():.1f} h")

check_budget("Dataset & budget done")

## §3 Proprietary Model Architectures (A / B / C / D / E) — Built from raw `torch.nn`

| Model | Architecture | Purpose |
|-------|-------------|---------|
| **A** — `ResidualUNet3D` | 3D nnUNet-style encoder–decoder with residual blocks, InstanceNorm3D, LeakyReLU, deep supervision (4-level, base=32) | Local continuity, stability, calibration anchor |
| **B** — `TransBottleneckUNet3D` | Same encoder/decoder + lightweight multi-head self-attention at bottleneck (2 transformer layers, 4-level, base=32) | Global coherence, fewer topology splits |
| **C** — `DeepZResUNet3D` | Shallower pyramid (3-level, /8 bottleneck) but wider channels (base=40). Trained with Z-heavy patches (160, 96, 96) | Z-continuity specialist, catches slice-wise breaks A/B miss |
| **D** — `SqueezeExciteUNet3D` | ResidualUNet3D + Squeeze-Excitation channel attention after every encoder/decoder stage (4-level, base=32) | Channel-adaptive: re-weights features for thin surface emphasis |
| **E** — `AttnGateUNet3D` | UNet with attention gates on skip connections — spatial attention modulates encoder features at each decoder level (4-level, base=32) | Spatial attention: suppresses background noise, focuses on surface regions |

**Design rules:** Conv3D everywhere · Residual connections · InstanceNorm3D + LeakyReLU · Deep supervision during training (off at inference) · Same normalization across all 5 models (calibration-compatible) · `classifier_activation=None` → raw logits for ensemble

In [ ]:
# ============================================================
# §3-A  PROPRIETARY MODEL A — ResidualUNet3D  (nnUNet-style)
# ============================================================
# 100 % hand-written from torch.nn primitives.
# Conv3D · Residual blocks · InstanceNorm3D · LeakyReLU
# Deep supervision outputs during training.
# ============================================================

class ConvBlock3D(nn.Module):
    """Two Conv3D layers with InstanceNorm3D + LeakyReLU, plus residual skip."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_ch, affine=True)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_ch, affine=True)
        self.act   = nn.LeakyReLU(0.01, inplace=True)
        # 1×1 residual projection when channel counts differ
        self.skip  = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.act(self.norm2(self.conv2(x)))
        return x + res


class DownBlock3D(nn.Module):
    """Strided 2× down-sample → ConvBlock3D."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv3d(in_ch, in_ch, 2, stride=2, bias=False)
        self.block = ConvBlock3D(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.down(x))


class UpBlock3D(nn.Module):
    """Trilinear 2× upsample → concat skip → ConvBlock3D."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.block = ConvBlock3D(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))


class ResidualUNet3D(nn.Module):
    """
    Proprietary 3D Residual Encoder–Decoder (nnUNet-style).
    - 4-level encoder, bottleneck, 4-level decoder
    - Deep supervision heads at decoder levels 1-3 (training only)
    - Outputs raw logits (no softmax)
    """
    def __init__(self, in_ch=1, num_classes=2, base=32):
        super().__init__()
        C = base  # 32
        # ── Encoder ──
        self.enc0 = ConvBlock3D(in_ch, C)           # /1    32
        self.enc1 = DownBlock3D(C,     C * 2)        # /2    64
        self.enc2 = DownBlock3D(C * 2, C * 4)        # /4   128
        self.enc3 = DownBlock3D(C * 4, C * 8)        # /8   256
        # ── Bottleneck ──
        self.bottleneck = DownBlock3D(C * 8, C * 16) # /16  512
        # ── Decoder ──
        self.dec3 = UpBlock3D(C * 16, C * 8, C * 8)  # /8   256
        self.dec2 = UpBlock3D(C * 8,  C * 4, C * 4)  # /4   128
        self.dec1 = UpBlock3D(C * 4,  C * 2, C * 2)  # /2    64
        self.dec0 = UpBlock3D(C * 2,  C,     C)      # /1    32
        # ── Head ──
        self.head = nn.Conv3d(C, num_classes, 1)
        # ── Deep supervision (training only) ──
        self.ds3 = nn.Conv3d(C * 8, num_classes, 1)
        self.ds2 = nn.Conv3d(C * 4, num_classes, 1)
        self.ds1 = nn.Conv3d(C * 2, num_classes, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out",
                                        nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision=False):
        # Encoder
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        # Bottleneck
        bn = self.bottleneck(e3)
        # Decoder
        d3 = self.dec3(bn, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        d0 = self.dec0(d1, e0)
        # Main output — raw logits
        out = self.head(d0)

        if deep_supervision and self.training:
            target_size = out.shape[2:]
            ds3 = F.interpolate(self.ds3(d3), size=target_size, mode="trilinear", align_corners=False)
            ds2 = F.interpolate(self.ds2(d2), size=target_size, mode="trilinear", align_corners=False)
            ds1 = F.interpolate(self.ds1(d1), size=target_size, mode="trilinear", align_corners=False)
            return out, ds3, ds2, ds1

        return out


# Quick sanity check
_tmp = ResidualUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
_p = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f"[OK] Model A (ResidualUNet3D): {_p:.2f} M params")
del _tmp

In [ ]:
# ============================================================
# §3-B  PROPRIETARY MODEL B — TransBottleneckUNet3D
# ============================================================
# Same encoder / decoder backbone as Model A, but with
# a lightweight multi-head self-attention block at the
# bottleneck (2 transformer layers, 4 heads).
# Bottleneck spatial size is small (/16) → OOM-safe.
# ============================================================

class TransformerBlock3D(nn.Module):
    """
    Single transformer layer operating on flattened 3-D spatial tokens.
    Pre-norm style (LayerNorm → MHSA → residual → LayerNorm → FFN → residual).
    """
    def __init__(self, dim, n_heads=4, ffn_ratio=2.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, n_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden     = int(dim * ffn_ratio)
        self.ffn   = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden, dim),
            nn.Dropout(drop),
        )

    def forward(self, x):
        # x: (B, N, C)  where N = D'×H'×W' tokens
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


class TransformerBottleneck(nn.Module):
    """Stack of TransformerBlock3D layers applied to the bottleneck feature map."""
    def __init__(self, channels, n_layers=2, n_heads=4):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock3D(channels, n_heads=n_heads)
            for _ in range(n_layers)
        ])

    def forward(self, x):
        # x: (B, C, D, H, W) → flatten spatial → transformer → reshape
        B, C, D, H, W = x.shape
        tokens = x.flatten(2).permute(0, 2, 1)       # (B, N, C)
        for layer in self.layers:
            tokens = layer(tokens)
        return tokens.permute(0, 2, 1).view(B, C, D, H, W)


class TransBottleneckUNet3D(nn.Module):
    """
    Model B: identical encoder/decoder to ResidualUNet3D, plus a
    transformer bottleneck for global coherence.
    - 2 transformer layers, 4 heads at /16 spatial resolution
    - Same InstanceNorm3D + LeakyReLU conventions as Model A
    - Deep supervision during training, raw logits output
    """
    def __init__(self, in_ch=1, num_classes=2, base=32, trans_layers=2, trans_heads=4):
        super().__init__()
        C = base
        # ── Encoder (shared architecture with Model A) ──
        self.enc0 = ConvBlock3D(in_ch, C)
        self.enc1 = DownBlock3D(C,     C * 2)
        self.enc2 = DownBlock3D(C * 2, C * 4)
        self.enc3 = DownBlock3D(C * 4, C * 8)
        # ── Bottleneck: Conv + Transformer ──
        self.bottleneck_conv = DownBlock3D(C * 8, C * 16)
        self.bottleneck_tx   = TransformerBottleneck(C * 16, n_layers=trans_layers, n_heads=trans_heads)
        # ── Decoder ──
        self.dec3 = UpBlock3D(C * 16, C * 8, C * 8)
        self.dec2 = UpBlock3D(C * 8,  C * 4, C * 4)
        self.dec1 = UpBlock3D(C * 4,  C * 2, C * 2)
        self.dec0 = UpBlock3D(C * 2,  C,     C)
        # ── Head ──
        self.head = nn.Conv3d(C, num_classes, 1)
        # ── Deep supervision ──
        self.ds3 = nn.Conv3d(C * 8, num_classes, 1)
        self.ds2 = nn.Conv3d(C * 4, num_classes, 1)
        self.ds1 = nn.Conv3d(C * 2, num_classes, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision=False):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        bn = self.bottleneck_conv(e3)
        bn = self.bottleneck_tx(bn)            # Transformer at bottleneck
        d3 = self.dec3(bn, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        d0 = self.dec0(d1, e0)
        out = self.head(d0)

        if deep_supervision and self.training:
            sz = out.shape[2:]
            ds3 = F.interpolate(self.ds3(d3), size=sz, mode="trilinear", align_corners=False)
            ds2 = F.interpolate(self.ds2(d2), size=sz, mode="trilinear", align_corners=False)
            ds1 = F.interpolate(self.ds1(d1), size=sz, mode="trilinear", align_corners=False)
            return out, ds3, ds2, ds1
        return out


# Sanity check
_tmp = TransBottleneckUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
_p = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f"[OK] Model B (TransBottleneckUNet3D): {_p:.2f} M params")
del _tmp

check_budget("Models defined")

In [ ]:
# ============================================================
# §3-C  PROPRIETARY MODEL C — DeepZResUNet3D  (Z-specialist)
# ============================================================
# Z-heavy diversity model: SHALLOWER pyramid (3 levels, not 4)
# but WIDER channels (base=40 vs 32).  Trained with Z-biased
# patches (160, 96, 96) to learn depth-continuity features.
#
# Why this is genuinely diverse:
#   - /8 bottleneck (vs /16 for A/B) → more spatial detail retained
#   - Wider channels → richer per-level representations
#   - Z-heavy training → weights biased toward depth continuity
#   - Catches slice-wise breaks that A/B miss
# ============================================================

class DeepZResUNet3D(nn.Module):
    """
    Model C: Z-specialist with 3-level encoder (shallower, wider).
    - 3-level encoder (/1 → /2 → /4), bottleneck at /8
    - base=40 channels (wider than A/B's 32)
    - Deep supervision at decoder levels 1-2
    - Same ConvBlock3D/DownBlock3D/UpBlock3D building blocks
    """

    def __init__(self, in_ch=1, num_classes=2, base=40):
        super().__init__()
        C = base  # 40

        # ── Encoder (3 levels — shallower than A/B's 4) ──
        self.enc0 = ConvBlock3D(in_ch, C)              # /1   40
        self.enc1 = DownBlock3D(C,     C * 2)           # /2   80
        self.enc2 = DownBlock3D(C * 2, C * 4)           # /4  160

        # ── Bottleneck at /8 (not /16 like A/B) ──
        self.bottleneck = DownBlock3D(C * 4, C * 8)     # /8  320

        # ── Decoder ──
        self.dec2 = UpBlock3D(C * 8, C * 4, C * 4)     # /4  160
        self.dec1 = UpBlock3D(C * 4, C * 2, C * 2)     # /2   80
        self.dec0 = UpBlock3D(C * 2, C,     C)          # /1   40

        # ── Head ──
        self.head = nn.Conv3d(C, num_classes, 1)

        # ── Deep supervision (training only, 2 aux heads) ──
        self.ds2 = nn.Conv3d(C * 4, num_classes, 1)
        self.ds1 = nn.Conv3d(C * 2, num_classes, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out",
                                        nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision=False):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        bn = self.bottleneck(e2)

        d2 = self.dec2(bn, e2)
        d1 = self.dec1(d2, e1)
        d0 = self.dec0(d1, e0)
        out = self.head(d0)

        if deep_supervision and self.training:
            sz = out.shape[2:]
            ds2 = F.interpolate(self.ds2(d2), size=sz, mode="trilinear",
                                align_corners=False)
            ds1 = F.interpolate(self.ds1(d1), size=sz, mode="trilinear",
                                align_corners=False)
            return out, ds2, ds1
        return out


# Sanity check
_tmp = DeepZResUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=MODEL_C_BASE)
_p = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f"[OK] Model C (DeepZResUNet3D): {_p:.2f} M params  |  base={MODEL_C_BASE}, levels=3, bottleneck=/8")
del _tmp

check_budget("All models defined")

In [ ]:
# ============================================================
# §3-D  PROPRIETARY MODEL D — SqueezeExciteUNet3D (channel attention)
# ============================================================
# ResidualUNet3D backbone + Squeeze-and-Excitation blocks after
# every encoder/decoder stage.  SE learns to re-weight channels
# to emphasise task-relevant features — especially effective for
# thin-surface detection where certain feature channels carry
# disproportionate signal.
# ============================================================

class SE3D(nn.Module):
    """Squeeze-and-Excitation block for 3D: learns channel re-weighting."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c = x.shape[:2]
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1, 1)
        return x * w


class SqueezeExciteUNet3D(nn.Module):
    """
    Model D: ResidualUNet3D + SE blocks after every encoder/decoder stage.
    Channel attention lets the network emphasise task-relevant features.
    4-level, base=32, ~19M params + negligible SE overhead.
    """
    def __init__(self, in_ch=1, num_classes=2, base=32):
        super().__init__()
        C = base
        self.enc0 = ConvBlock3D(in_ch, C);        self.se_e0 = SE3D(C)
        self.enc1 = DownBlock3D(C, C * 2);        self.se_e1 = SE3D(C * 2)
        self.enc2 = DownBlock3D(C * 2, C * 4);    self.se_e2 = SE3D(C * 4)
        self.enc3 = DownBlock3D(C * 4, C * 8);    self.se_e3 = SE3D(C * 8)
        self.bottleneck = DownBlock3D(C * 8, C * 16); self.se_bn = SE3D(C * 16)
        self.dec3 = UpBlock3D(C * 16, C * 8, C * 8);  self.se_d3 = SE3D(C * 8)
        self.dec2 = UpBlock3D(C * 8, C * 4, C * 4);   self.se_d2 = SE3D(C * 4)
        self.dec1 = UpBlock3D(C * 4, C * 2, C * 2);   self.se_d1 = SE3D(C * 2)
        self.dec0 = UpBlock3D(C * 2, C, C);            self.se_d0 = SE3D(C)
        self.head = nn.Conv3d(C, num_classes, 1)
        self.ds3 = nn.Conv3d(C * 8, num_classes, 1)
        self.ds2 = nn.Conv3d(C * 4, num_classes, 1)
        self.ds1 = nn.Conv3d(C * 2, num_classes, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision=False):
        e0 = self.se_e0(self.enc0(x))
        e1 = self.se_e1(self.enc1(e0))
        e2 = self.se_e2(self.enc2(e1))
        e3 = self.se_e3(self.enc3(e2))
        bn = self.se_bn(self.bottleneck(e3))
        d3 = self.se_d3(self.dec3(bn, e3))
        d2 = self.se_d2(self.dec2(d3, e2))
        d1 = self.se_d1(self.dec1(d2, e1))
        d0 = self.se_d0(self.dec0(d1, e0))
        out = self.head(d0)
        if deep_supervision and self.training:
            sz = out.shape[2:]
            ds3 = F.interpolate(self.ds3(d3), size=sz, mode="trilinear", align_corners=False)
            ds2 = F.interpolate(self.ds2(d2), size=sz, mode="trilinear", align_corners=False)
            ds1 = F.interpolate(self.ds1(d1), size=sz, mode="trilinear", align_corners=False)
            return out, ds3, ds2, ds1
        return out


# ============================================================
# §3-E  PROPRIETARY MODEL E — AttnGateUNet3D (spatial attention on skips)
# ============================================================
# Standard UNet backbone with attention gates on skip connections.
# Spatial attention helps the decoder focus on relevant encoder
# features, suppressing background noise — particularly effective
# for thin surfaces embedded in noisy CT volumes.
# ============================================================

class AttentionGate3D(nn.Module):
    """Attention gate: gate signal from deeper layer modulates skip features."""
    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Conv3d(gate_ch, inter_ch, 1, bias=False)
        self.W_x = nn.Conv3d(skip_ch, inter_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.Conv3d(inter_ch, 1, 1, bias=False),
            nn.Sigmoid()
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = F.interpolate(self.W_g(g), size=x.shape[2:], mode="trilinear", align_corners=False)
        x1 = self.W_x(x)
        alpha = self.psi(self.act(g1 + x1))
        return x * alpha


class AttnGateUNet3D(nn.Module):
    """
    Model E: UNet with attention gates on skip connections.
    Spatial attention helps the decoder focus on relevant encoder features,
    suppressing irrelevant activations — especially useful for thin surfaces.
    4-level, base=32, ~20M params.
    """
    def __init__(self, in_ch=1, num_classes=2, base=32):
        super().__init__()
        C = base
        self.enc0 = ConvBlock3D(in_ch, C)
        self.enc1 = DownBlock3D(C, C * 2)
        self.enc2 = DownBlock3D(C * 2, C * 4)
        self.enc3 = DownBlock3D(C * 4, C * 8)
        self.bottleneck = DownBlock3D(C * 8, C * 16)
        self.ag3 = AttentionGate3D(C * 16, C * 8, C * 4)
        self.ag2 = AttentionGate3D(C * 8, C * 4, C * 2)
        self.ag1 = AttentionGate3D(C * 4, C * 2, C)
        self.ag0 = AttentionGate3D(C * 2, C, max(C // 2, 1))
        self.dec3 = UpBlock3D(C * 16, C * 8, C * 8)
        self.dec2 = UpBlock3D(C * 8, C * 4, C * 4)
        self.dec1 = UpBlock3D(C * 4, C * 2, C * 2)
        self.dec0 = UpBlock3D(C * 2, C, C)
        self.head = nn.Conv3d(C, num_classes, 1)
        self.ds3 = nn.Conv3d(C * 8, num_classes, 1)
        self.ds2 = nn.Conv3d(C * 4, num_classes, 1)
        self.ds1 = nn.Conv3d(C * 2, num_classes, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision=False):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        bn = self.bottleneck(e3)
        d3 = self.dec3(bn, self.ag3(bn, e3))
        d2 = self.dec2(d3, self.ag2(d3, e2))
        d1 = self.dec1(d2, self.ag1(d2, e1))
        d0 = self.dec0(d1, self.ag0(d1, e0))
        out = self.head(d0)
        if deep_supervision and self.training:
            sz = out.shape[2:]
            ds3 = F.interpolate(self.ds3(d3), size=sz, mode="trilinear", align_corners=False)
            ds2 = F.interpolate(self.ds2(d2), size=sz, mode="trilinear", align_corners=False)
            ds1 = F.interpolate(self.ds1(d1), size=sz, mode="trilinear", align_corners=False)
            return out, ds3, ds2, ds1
        return out


# Sanity checks
_tmp = SqueezeExciteUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
_p = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f"[OK] Model D (SqueezeExciteUNet3D): {_p:.2f} M params")
del _tmp

_tmp = AttnGateUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
_p = sum(p.numel() for p in _tmp.parameters()) / 1e6
print(f"[OK] Model E (AttnGateUNet3D): {_p:.2f} M params")
del _tmp

check_budget("All 5 models defined")

## §3 (cont.) Training — Proprietary Loss, Dataset, EMA, Deep Supervision

**Loss:** Dice + Cross-Entropy + **Boundary Dice** (targets SurfaceDice@τ directly)  
**Dataset:** Random 3D patch extraction with foreground oversampling (70 % FG patches)  
**EMA:** Exponential moving average of weights for inference stability  
**Deep supervision:** Auxiliary losses at decoder levels 3, 2, 1 (weighted 0.5, 0.25, 0.125)  
**Budget gate:** Training auto-stops when `MAX_TRAIN_HOURS` exceeded

In [ ]:
# ============================================================
# §3 (cont.)  TRAINING — LOSS, DATASET, EMA, LOOP
# ============================================================

# ─────────────────────────────────────────────────────────────
# Loss: Dice + CE + Boundary Dice  (targets SurfaceDice@τ + topology)
# ─────────────────────────────────────────────────────────────

class BoundaryDiceLoss(nn.Module):
    """
    Dice loss computed only on boundary (surface) voxels.
    Directly targets SurfaceDice@τ by teaching the model to place
    surfaces precisely.  Boundaries extracted via 3D morphological
    erosion (min-pool through negated max-pool) — fully differentiable.
    """
    def __init__(self, smooth=1e-5, kernel_size=3):
        super().__init__()
        self.smooth = smooth
        self.ks = kernel_size

    def _extract_boundary(self, x):
        """x: (B, 1, D, H, W) float [0,1] → boundary map."""
        pad = self.ks // 2
        eroded = -F.max_pool3d(-x, self.ks, stride=1, padding=pad)
        return (x - eroded).clamp(0, 1)

    def forward(self, logits, target):
        probs = torch.softmax(logits, dim=1)
        pred_fg = probs[:, 1:2]                            # (B, 1, D, H, W)
        gt_fg = (target == 1).float().unsqueeze(1)          # (B, 1, D, H, W)
        pred_bd = self._extract_boundary(pred_fg)
        gt_bd   = self._extract_boundary(gt_fg)
        inter = (pred_bd * gt_bd).sum()
        union = pred_bd.sum() + gt_bd.sum()
        return 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)


class DiceCELoss(nn.Module):
    """
    Dice + Cross-Entropy + Boundary Dice.
    - CE:   voxel-wise accuracy
    - Dice: region overlap
    - Boundary Dice: surface accuracy (targets SurfaceDice@τ)
    Works on raw logits.
    """
    def __init__(self, num_classes=2, smooth=1e-5,
                 ce_weight=1.0, dice_weight=1.0, boundary_weight=0.5):
        super().__init__()
        self.smooth = smooth
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.boundary_weight = boundary_weight
        self.ce = nn.CrossEntropyLoss()
        self.boundary_dice = BoundaryDiceLoss(smooth=smooth)

    def forward(self, logits, target):
        # logits: (B, C, D, H, W)   target: (B, D, H, W) long
        ce_loss = self.ce(logits, target)

        # Dice per class
        probs = torch.softmax(logits, dim=1)
        target_oh = F.one_hot(target, logits.shape[1]).permute(0, 4, 1, 2, 3).float()
        dims = (0, 2, 3, 4)
        inter = (probs * target_oh).sum(dims)
        union = probs.sum(dims) + target_oh.sum(dims)
        dice = 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)
        dice_loss = dice.mean()

        # Boundary Dice (surface-aware)
        boundary_loss = self.boundary_dice(logits, target)

        return (self.ce_weight * ce_loss +
                self.dice_weight * dice_loss +
                self.boundary_weight * boundary_loss)


# ─────────────────────────────────────────────────────────────
# Dataset: random 3D patch extraction with FG oversampling
# ─────────────────────────────────────────────────────────────
class VesuviusPatchDataset(Dataset):
    """
    Single-volume lazy cache — keeps only ONE volume in RAM at a time.
    - Consecutive indices (idx // ppv) map to the same volume → cache hit
    - Random crops + FG oversampling provide per-patch diversity
    - Requires num_workers=0 & shuffle=False so index order is sequential
    - Memory: ~1 volume in RAM (~200-800 MB) instead of all volumes
    """
    def __init__(self, vol_infos, patch_size=(128, 128, 128), patches_per_vol=8, fg_rate=0.7):
        self.vol_infos = vol_infos
        self.ps = patch_size
        self.ppv = patches_per_vol
        self.fg_rate = fg_rate
        self.length = len(vol_infos) * patches_per_vol
        # Single-volume cache (loaded on first access, re-used for ppv patches)
        self._cached_vol_idx = -1
        self._cached_vol = None
        self._cached_lbl = None
        print(f"[DATASET] Lazy single-volume cache: {len(vol_infos)} vols × "
              f"{patches_per_vol} patches = {self.length} samples/epoch")

    def __len__(self):
        return self.length

    @staticmethod
    def _norm(vol):
        mask = vol > 0
        if mask.any():
            mu, sigma = vol[mask].mean(), vol[mask].std() + 1e-8
            vol = (vol - mu) / sigma
        return vol

    def _load_volume(self, vol_idx):
        """Load volume with single-entry cache — avoids repeated disk I/O."""
        if vol_idx == self._cached_vol_idx:
            return self._cached_vol, self._cached_lbl
        info = self.vol_infos[vol_idx % len(self.vol_infos)]
        vol = read_tif_volume(info["image"]).astype(np.float32)
        lbl = read_tif_volume(info["label"]).astype(np.int64)
        lbl = (lbl > 0).astype(np.int64)
        vol = self._norm(vol)
        # Pad if smaller than patch
        for ax, ps in enumerate(self.ps):
            if vol.shape[ax] < ps:
                pad_before = (ps - vol.shape[ax]) // 2
                pad_after = ps - vol.shape[ax] - pad_before
                pads = [(0, 0)] * 3
                pads[ax] = (pad_before, pad_after)
                vol = np.pad(vol, pads, mode="constant")
                lbl = np.pad(lbl, pads, mode="constant")
        mb = (vol.nbytes + lbl.nbytes) / 1e6
        print(f"  [CACHE] Loaded vol {vol_idx}: {info['image']} "
              f"shape={vol.shape} ({mb:.0f} MB)")
        # Evict old, store new
        self._cached_vol_idx = vol_idx
        self._cached_vol = vol
        self._cached_lbl = lbl
        return vol, lbl

    def _random_crop(self, vol, lbl, require_fg=False):
        D, H, W = vol.shape
        pd, ph, pw = self.ps
        max_tries = 20 if require_fg else 1
        for _ in range(max_tries):
            d0 = random.randint(0, max(0, D - pd))
            h0 = random.randint(0, max(0, H - ph))
            w0 = random.randint(0, max(0, W - pw))
            patch_v = vol[d0:d0+pd, h0:h0+ph, w0:w0+pw]
            patch_l = lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]
            if not require_fg or patch_l.any():
                return patch_v, patch_l
        return patch_v, patch_l   # fallback

    def __getitem__(self, idx):
        vol_idx = idx // self.ppv
        vol, lbl = self._load_volume(vol_idx)
        require_fg = random.random() < self.fg_rate
        pv, pl = self._random_crop(vol, lbl, require_fg=require_fg)
        # (1, D, H, W) tensor
        return torch.from_numpy(pv[None].copy()), torch.from_numpy(pl.copy())


# ─────────────────────────────────────────────────────────────
# EMA wrapper
# ─────────────────────────────────────────────────────────────
class EMAModel:
    """Exponential moving average of model parameters for stable inference."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v, alpha=1.0 - self.decay)

    def apply(self, model):
        model.load_state_dict(self.shadow)


# ─────────────────────────────────────────────────────────────
# Training loop (budget-gated)
# ─────────────────────────────────────────────────────────────
DS_WEIGHTS = [1.0, 0.5, 0.25, 0.125]   # main, ds3, ds2, ds1

def train_one_model(model, tag="Model", patch_size=None, max_hours=None):
    """
    Train a single model with:
      - Mixed-precision (AMP)
      - Deep supervision
      - EMA
      - Per-model time budget (max_hours) AND global budget gate
    Returns (model_with_ema_weights, train_losses).
    """
    if patch_size is None:
        patch_size = TRAIN_PATCH
    if max_hours is None:
        max_hours = MAX_TRAIN_HOURS / 5   # equal split across 5 models
    model = model.to(DEVICE)
    criterion = DiceCELoss(num_classes=NUM_CLASSES, boundary_weight=PP_BOUNDARY_W).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-6)
    scaler = GradScaler("cuda")
    ema = EMAModel(model)
    ds = VesuviusPatchDataset(train_vol_ids, patch_size=patch_size,
                              patches_per_vol=8, fg_rate=FG_OVERSAMPLE)
    # num_workers=0 + shuffle=False: sequential indices hit single-volume
    # cache (15/16 cache hits per volume). Random crops provide diversity.
    loader = DataLoader(ds, batch_size=TRAIN_BATCH, shuffle=False,
                        num_workers=0, pin_memory=True, drop_last=True)

    train_start = time.time()
    losses = []

    for epoch in range(TRAIN_EPOCHS):
        # Per-model budget gate
        model_hours = (time.time() - train_start) / 3600
        if model_hours > max_hours:
            print(f"[BUDGET] {tag} budget exhausted at epoch {epoch} ({model_hours:.2f}h / {max_hours:.2f}h)")
            break
        # Global pipeline budget gate
        if (time.time() - pipeline_start) / 3600 > MAX_TRAIN_HOURS:
            print(f"[BUDGET] Global training budget exhausted at epoch {epoch}")
            break

        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for img, lbl in loader:
            img, lbl = img.to(DEVICE), lbl.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                outputs = model(img, deep_supervision=True)
                if isinstance(outputs, tuple):
                    loss = sum(w * criterion(o, lbl)
                               for w, o in zip(DS_WEIGHTS, outputs))
                else:
                    loss = criterion(outputs, lbl)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 12.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)

            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg = epoch_loss / max(n_batches, 1)
        losses.append(avg)
        elapsed_m = (time.time() - train_start) / 60
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{tag}] Epoch {epoch+1:3d}/{TRAIN_EPOCHS}  loss={avg:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed_m:.0f} min)")

    # Apply EMA weights
    ema.apply(model)
    model.eval()

    # Save checkpoint
    ckpt_path = f"{CKPT_DIR}/{tag.lower().replace(' ', '_')}.pt"
    torch.save(model.state_dict(), ckpt_path)
    final_msg = f"final loss={losses[-1]:.4f}" if losses else "no training (budget exhausted)"
    print(f"[OK] {tag} saved \u2192 {ckpt_path}  ({len(losses)} epochs, {final_msg})")
    return model, losses


# ── Execute training / loading ──
model_a = ResidualUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
model_b = TransBottleneckUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
model_c = DeepZResUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=MODEL_C_BASE)
model_d = SqueezeExciteUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)
model_e = AttnGateUNet3D(in_ch=1, num_classes=NUM_CLASSES, base=BASE_CHANNELS)

_all_models_tags = [
    ("model_a", model_a, {}),
    ("model_b", model_b, {}),
    ("model_c", model_c, {"patch_size": MODEL_C_PATCH}),
    ("model_d", model_d, {}),
    ("model_e", model_e, {}),
]

if TRAIN_ENABLED and len(train_vol_ids) > 0:
    print(f"\n{'='*60}")
    print("[TRAIN] Starting proprietary training pipeline (5 models)")
    print(f"{'='*60}")
    check_budget("Training start")

    for _tag, _mdl, _kw in _all_models_tags:
        _mdl, _losses = train_one_model(_mdl, tag=_tag, **_kw)
        check_budget(f"{_tag} trained")
        torch.cuda.empty_cache(); gc.collect()

    print(f"\n[OK] All 5 models trained")
else:
    # Attempt to load pre-trained weights (PRETRAINED_DIR first, then CKPT_DIR)
    for tag, mdl, _ in _all_models_tags:
        loaded = False
        for search_dir in [PRETRAINED_DIR, CKPT_DIR]:
            ckpt = f"{search_dir}/{tag}.pt"
            if os.path.exists(ckpt):
                mdl.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
                print(f"[OK] Loaded {tag} from {ckpt}")
                loaded = True
                break
        if not loaded:
            print(f"[WARN] No checkpoint for {tag} \u2014 model will use random init")
    for _, mdl, _ in _all_models_tags:
        mdl.to(DEVICE).eval()

# 3-model ensemble: D (best), C (2nd), A (3rd)
models = [model_d]  # best: loss=3.43
if ENSEMBLE_MODE:
    models.extend([model_c, model_a])  # C=3.52, A=4.11
ENSEMBLE_WEIGHTS = [0.40, 0.35, 0.25]  # [D, C, A]

active_weights = ENSEMBLE_WEIGHTS[:len(models)]
print(f"\n[OK] Ensemble: {len(models)} models, method={ENSEMBLE_METHOD}, weights={active_weights}")

check_budget("Training / loading complete")

## §5–7 Inference Engine — Sliding Window + Gaussian Blending + Degradation

**Degradation ladder** (7 levels, triggered by OOM or time pressure):
1. Full quality (TTA + high overlap + full patch + all 3 models)
2. Disable TTA
3. Reduce overlap
4. Reduce Z patch dimension
5. Reduce all patch dimensions
6. **Minimum patch + drop Model C** (2 models only)
7. **Minimum patch + Model A only** (single model fallback)

**Time-aware guard:** Before each volume, the loop checks remaining wall-clock budget. If seconds-per-volume falls below 70% of `MAX_SEC_PER_VOL`, it proactively degrades — including model dropping — to guarantee every volume gets processed.

**Never skip:** Even total failure → zero-mask fallback with correct shape (from pre-cached metadata).

In [ ]:
# ============================================================
# §5-7  SLIDING WINDOW INFERENCE, DEGRADATION, TTA
# ============================================================

# ─────────────────────────────────────────────────────────────
# Degradation Ladder (OOM-resistant)
# ─────────────────────────────────────────────────────────────
class DegradationState:
    """
    8-level degradation ladder for 5-model ensemble.
    Levels 0-1: full/basic TTA + high overlap + all 5 models
    Levels 2-4: reduce TTA/overlap/patch, keep 5 models
    Levels 5-7: reduce models progressively
    """
    LEVELS = [
        {"tta": "basic", "overlap": 0.35, "roi": (128, 128, 128), "max_models": 3, "label": "Basic TTA, 3 models"},
        {"tta": False,   "overlap": 0.35, "roi": (128, 128, 128), "max_models": 3, "label": "No TTA, 3 models"},
        {"tta": False,   "overlap": 0.25, "roi": (128, 128, 128), "max_models": 3, "label": "Low overlap, 3 models"},
        {"tta": False,   "overlap": 0.25, "roi": (128, 128, 128), "max_models": 2, "label": "Low overlap, 2 models"},
        {"tta": False,   "overlap": 0.20, "roi": (96,  96,  96),  "max_models": 1, "label": "Small + best only"},
        {"tta": False,   "overlap": 0.15, "roi": (64,  64,  64),  "max_models": 1, "label": "Min + best only"},
    ]
    def __init__(self):
        self.level = 0  # starts at no-TTA level (TTA disabled globally)
    def current(self):
        return self.LEVELS[min(self.level, len(self.LEVELS) - 1)]
    def degrade(self):
        if self.level < len(self.LEVELS) - 1:
            self.level += 1
            print(f"[DEGRADE] → level {self.level}: {self.current()['label']}")
        else:
            print("[DEGRADE] Already at minimum")
        return self.current()

degradation = DegradationState()


# ─────────────────────────────────────────────────────────────
# Gaussian importance map (3-D) for blending
# ─────────────────────────────────────────────────────────────
def _gaussian_importance(patch_size, sigma_scale=0.125):
    """Create 3-D Gaussian weight map, peaks at centre, tapers at edges."""
    maps = []
    for s in patch_size:
        ax = np.arange(s, dtype=np.float32)
        g = np.exp(-0.5 * ((ax - s / 2) / (s * sigma_scale)) ** 2)
        maps.append(g)
    w = maps[0][:, None, None] * maps[1][None, :, None] * maps[2][None, None, :]
    w = np.clip(w, 1e-6, None)
    return w


# ─────────────────────────────────────────────────────────────
# Proprietary Sliding-Window Inference (pure NumPy + PyTorch)
# ─────────────────────────────────────────────────────────────
@torch.no_grad()
def sliding_window_inference(model, volume_np, roi_size, overlap, num_classes):
    """
    Patch-wise inference with Gaussian blending.

    Args:
        model      : nn.Module on DEVICE (eval mode)
        volume_np  : np.ndarray  (D, H, W)  float32, already normalised
        roi_size   : tuple (pd, ph, pw)
        overlap    : float 0..1
        num_classes: int
    Returns:
        logits_volume : np.ndarray (D, H, W, num_classes) float32
    """
    D, H, W = volume_np.shape
    pd, ph, pw = roi_size

    # Pad volume so every patch fits
    pad_d = max(0, pd - D)
    pad_h = max(0, ph - H)
    pad_w = max(0, pw - W)
    vol = np.pad(volume_np,
                 ((0, pad_d), (0, pad_h), (0, pad_w)),
                 mode="constant")
    Dp, Hp, Wp = vol.shape

    # Gaussian blending weights
    gauss = _gaussian_importance(roi_size)

    # Output accumulators
    logit_sum = np.zeros((num_classes, Dp, Hp, Wp), dtype=np.float32)
    weight_sum = np.zeros((Dp, Hp, Wp), dtype=np.float32)

    # Compute strides
    step = lambda total, patch, olap: max(1, int(patch * (1.0 - olap)))
    sd, sh, sw = step(Dp, pd, overlap), step(Hp, ph, overlap), step(Wp, pw, overlap)

    # Sliding window
    for d0 in range(0, Dp - pd + 1, sd):
        for h0 in range(0, Hp - ph + 1, sh):
            for w0 in range(0, Wp - pw + 1, sw):
                patch = vol[d0:d0+pd, h0:h0+ph, w0:w0+pw]
                inp = torch.from_numpy(patch[None, None]).to(DEVICE)  # (1, 1, D, H, W)

                with autocast("cuda"):
                    out = model(inp)  # (1, C, D, H, W) raw logits
                out = out.float().cpu().numpy()[0]  # (C, pd, ph, pw)

                logit_sum[:, d0:d0+pd, h0:h0+ph, w0:w0+pw] += out * gauss[None]
                weight_sum[d0:d0+pd, h0:h0+ph, w0:w0+pw] += gauss

    # Normalise
    weight_sum = np.maximum(weight_sum, 1e-8)
    logits = logit_sum / weight_sum[None]

    # Remove padding, transpose to (D, H, W, C)
    logits = logits[:, :D, :H, :W]
    return logits.transpose(1, 2, 3, 0)   # (D, H, W, C)


# ─────────────────────────────────────────────────────────────
# Test-Time Augmentation — enhanced 8-way + basic 4-way, returns LOGITS
# ─────────────────────────────────────────────────────────────
def predict_with_tta(model, volume_np, roi_size, overlap, num_classes, tta_mode="basic"):
    """
    Enhanced TTA with 3 modes:
      'full'  — all 8 flip combinations (2³): original + 7 flipped views
      'basic' — original + 3 single-axis flips (4 views, faster)
      False   — no augmentation (1 view only)
    Returns (D, H, W, C) raw logits.
    """
    logits = sliding_window_inference(model, volume_np, roi_size, overlap, num_classes)
    n = 1
    if tta_mode == "full":
        for axes in [(0,), (1,), (2,), (0,1), (0,2), (1,2), (0,1,2)]:
            flipped = np.flip(volume_np, axis=axes).copy()
            pred = sliding_window_inference(model, flipped, roi_size, overlap, num_classes)
            pred = np.flip(pred, axis=axes).copy()
            logits = logits + pred
            n += 1
    elif tta_mode == "basic":
        for axis in [0, 1, 2]:  # D, H, W
            flipped = np.flip(volume_np, axis=axis).copy()
            pred = sliding_window_inference(model, flipped, roi_size, overlap, num_classes)
            pred = np.flip(pred, axis=axis).copy()
            logits = logits + pred
            n += 1
    return logits / n


# ─────────────────────────────────────────────────────────────
# Volume loading + normalisation
# ─────────────────────────────────────────────────────────────
def load_test_volume(path):
    """Load + z-score normalisation on nonzero voxels."""
    vol = read_tif_volume(path).astype(np.float32)
    assert vol.ndim == 3, f"[FATAL] Expected 3D volume, got {vol.ndim}D with shape {vol.shape}"
    mask = vol > 0
    if mask.any():
        mu, sigma = vol[mask].mean(), vol[mask].std() + 1e-8
        vol = (vol - mu) / sigma

    return vol


check_budget("Inference utilities ready")
print("[OK] Sliding-window inference, TTA, degradation ladder ready")


## §8 Postprocessing — Topology-Aware for Competition Metric

**Metric:** `Score = 0.30×TopoScore + 0.35×SurfaceDice@τ=2 + 0.35×VOI_score`

**Strategy:** Bridges across wraps are the #1 killer (VOI merge↓ + TopoScore k=0/k=1↓). Favor under-segmentation over bridges.

| Step | Method | Purpose |
|------|--------|--------|
| 8.1 | **Hysteresis thresholding** | `T_high=0.75` seeds + `T_low=0.40` continuation. Conservative to avoid bridge propagation |
| 8.2 | **Anisotropic 3D closing** | Minimal radius (Z=1, XY=1). Heal micro-cracks without bridging adjacent wraps |
| 8.3 | **Dust removal** | Remove components < 250 voxels. Stabilises VOI + TopoScore k=0 |
| 8.4 | **Relative component filter** | Remove components < 1% of largest. Cleans topology under 26-connectivity |

**Prohibited:** Global hole filling · Aggressive dilation · Large closing radii · Topology-blind morphology

In [ ]:
# ============================================================
# §8  POSTPROCESSING  —  Hysteresis + Anisotropic Closing + Dust
# ============================================================

def _anisotropic_struct(z_r, xy_r):
    """Build anisotropic 3-D structuring element."""
    if z_r == 0 and xy_r == 0:
        return None
    if z_r == 0:
        s = 2 * xy_r + 1
        st = np.zeros((1, s, s), dtype=bool)
        for dy in range(-xy_r, xy_r + 1):
            for dx in range(-xy_r, xy_r + 1):
                if dy * dy + dx * dx <= xy_r * xy_r:
                    st[0, xy_r + dy, xy_r + dx] = True
        return st
    if xy_r == 0:
        st = np.zeros((2 * z_r + 1, 1, 1), dtype=bool)
        st[:, 0, 0] = True
        return st
    d, s = 2 * z_r + 1, 2 * xy_r + 1
    st = np.zeros((d, s, s), dtype=bool)
    for dz in range(-z_r, z_r + 1):
        for dy in range(-xy_r, xy_r + 1):
            for dx in range(-xy_r, xy_r + 1):
                if dy * dy + dx * dx <= xy_r * xy_r:
                    st[z_r + dz, xy_r + dy, xy_r + dx] = True
    return st


def topo_postprocess(fg_probs, T_low=0.40, T_high=0.75,
                     z_radius=1, xy_radius=1, dust_min=250):
    """
    Topology-optimised postprocessing — tuned for competition metric:
      Score = 0.30×TopoScore + 0.35×SurfaceDice@τ=2 + 0.35×VOI_score

    Strategy: favor within-wrap continuity, aggressively avoid cross-wrap bridges.
    Bridges kill VOI (merge↓) and TopoScore (k=0/k=1↓).
    Splits hurt less than bridges, so err on the side of under-segmentation.

      8.1  Hysteresis thresholding (T_high seeds, T_low continuation)
      8.2  Anisotropic 3D closing  (minimal radius — avoid bridging wraps)
      8.3  Dust removal            (absolute + relative to largest component)

    Input:  fg_probs  float (D, H, W) in [0, 1]
    Output: uint8 mask {0, 1}
    """
    strong = fg_probs >= T_high
    weak   = fg_probs >= T_low
    if not strong.any():
        return np.zeros_like(fg_probs, dtype=np.uint8)

    # 8.1 Hysteresis: propagate from strong seeds through weak regions (26-conn)
    struct26 = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct26)
    if not mask.any():
        return np.zeros_like(fg_probs, dtype=np.uint8)

    # 8.2 Anisotropic closing (minimal — only heal 1-voxel micro-cracks)
    st = _anisotropic_struct(z_radius, xy_radius)
    if st is not None:
        mask = ndi.binary_closing(mask, structure=st)

    # 8.3 Dust removal — absolute threshold
    if dust_min > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min)

    # 8.4 Relative component filtering (26-connectivity, matches VOI eval)
    #     Remove components < 1% of the largest — cleans up topology
    if mask.any():
        labeled, nc = ndi.label(mask, structure=struct26)
        if nc > 1:
            sizes = np.bincount(labeled.ravel())[1:]  # skip background
            largest = sizes.max()
            rel_thresh = max(dust_min, int(largest * 0.01))
            for comp_id, sz in enumerate(sizes, 1):
                if sz < rel_thresh:
                    mask[labeled == comp_id] = False

    return mask.astype(np.uint8)


print("[OK] Postprocessing ready (hysteresis + closing + dust)")

## §9 Debug Visualizations (development only)

- Tri-panel slices, MIP 3-view, topology metrics
- Always behind `DEBUG_VIZ` flag — **never runs during submission**
- Always wrapped in `try / except`

In [ ]:
# ============================================================
# §9  DEBUG VISUALIZATIONS  (guarded by DEBUG_VIZ)
# ============================================================

def compute_topo_metrics(mask):
    try:
        labeled, nc = ndi.label(mask, structure=ndi.generate_binary_structure(3, 3))
        if nc == 0:
            return {"n_comp": 0, "largest_frac": 0.0, "dust": 0, "fg_pct": 0.0}
        sizes = np.bincount(labeled.ravel())[1:]
        tfg = mask.sum()
        return {
            "n_comp": nc,
            "largest_frac": float(sizes.max() / tfg) if tfg > 0 else 0.0,
            "dust": int((sizes < 100).sum()),
            "fg_pct": 100.0 * tfg / mask.size,
        }
    except Exception as e:
        print(f"[WARN] topo metrics failed: {e}")
        return {"n_comp": 0, "largest_frac": 0, "dust": 0, "fg_pct": 0}

def plot_mip(vol, mask, title="MIP"):
    try:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        for i, ax in enumerate(["Z", "Y", "X"]):
            axes[0, i].imshow(vol.max(axis=i), cmap="gray"); axes[0, i].set_title(f"Vol ({ax})"); axes[0, i].axis("off")
            axes[1, i].imshow(mask.max(axis=i), cmap="hot");  axes[1, i].set_title(f"Mask ({ax})"); axes[1, i].axis("off")
        plt.suptitle(title); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"[WARN] MIP failed: {e}")

def plot_slices(vol, mask, title="Slice"):
    try:
        s = vol.shape[0] // 2
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(vol[s], cmap="gray"); axes[0].set_title(f"Input (z={s})"); axes[0].axis("off")
        axes[1].imshow(mask[s], cmap="hot");  axes[1].set_title(f"Pred  (z={s})"); axes[1].axis("off")
        plt.suptitle(title); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"[WARN] slice plot failed: {e}")

def log_topo(m, sid):
    try:
        print(f"[TOPO] {sid}: {m['n_comp']} comp, largest={m['largest_frac']*100:.1f}%, "
              f"dust={m['dust']}, FG={m['fg_pct']:.2f}%")
    except Exception:
        pass

print(f"[OK] Viz functions loaded (DEBUG_VIZ={DEBUG_VIZ})")

## §10 Submission / §11 Validator / §12 Red-Flag Checks

**Per-volume streaming:**
Load → Ensemble inference (logit-level) → Softmax → Postprocess → Write `.tif` → ZIP → Delete temp → `gc.collect()`

**§11 Hard gate:** ZIP exists · Only `.tif` · Count == `len(test.csv)` · IDs match · Spot-check `dtype uint8`, values `{0,1}`

**§12 Red flags:** OOM · RAM creep · Missing IDs · Empty masks

In [ ]:
# ============================================================
# §9.5  SMOKE TEST — 1 fragment end-to-end dry run
# ============================================================
# Runs a single test volume through the full pipeline:
#   load → inference (1 model, no TTA, min patch) → postprocess → tif → zip → verify
# Proves the entire chain works before committing 8+ hours.
# ============================================================

print("[SMOKE] Starting mini end-to-end smoke test...")
_smoke_ok = False

def np_softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

try:
    _smoke_id = test_ids[0]
    _smoke_path = f"{TEST_DIR}/{_smoke_id}.tif"
    _smoke_vol = load_test_volume(_smoke_path)
    print(f"  Volume loaded: shape={_smoke_vol.shape}, dtype={_smoke_vol.dtype}")

    # Minimal inference: Model A only, no TTA, small overlap, small ROI
    _smoke_roi = tuple(min(s, 64) for s in _smoke_vol.shape)
    _smoke_logits = predict_with_tta(
        models[0], _smoke_vol, roi_size=_smoke_roi,
        overlap=0.15, num_classes=NUM_CLASSES, tta_mode=False
    )
    print(f"  Logits: shape={_smoke_logits.shape}, range=[{_smoke_logits.min():.2f}, {_smoke_logits.max():.2f}]")

    _smoke_probs = np_softmax(_smoke_logits, axis=-1)[..., 1]
    _smoke_mask = topo_postprocess(_smoke_probs, T_low=PP_T_LOW, T_high=PP_T_HIGH,
                                    z_radius=PP_Z_RADIUS, xy_radius=PP_XY_RADIUS,
                                    dust_min=PP_DUST_MIN)
    print(f"  Mask: shape={_smoke_mask.shape}, dtype={_smoke_mask.dtype}, "
          f"unique={np.unique(_smoke_mask).tolist()}, "
          f"FG={100.0 * _smoke_mask.sum() / _smoke_mask.size:.2f}%")

    assert _smoke_mask.shape == _smoke_vol.shape, "Shape mismatch!"
    assert _smoke_mask.dtype == np.uint8, f"Bad dtype: {_smoke_mask.dtype}"
    assert set(np.unique(_smoke_mask)).issubset({0, 1}), "Bad values!"

    # Write → zip → re-read → verify
    _smoke_tif = f"{OUTPUT_DIR}/_smoke_{_smoke_id}.tif"
    _smoke_zip = f"{OUTPUT_DIR}/_smoke_test.zip"
    tifffile.imwrite(_smoke_tif, _smoke_mask)
    with zipfile.ZipFile(_smoke_zip, "w") as _szf:
        _szf.write(_smoke_tif, arcname=f"{_smoke_id}.tif")
    with zipfile.ZipFile(_smoke_zip, "r") as _szf:
        with _szf.open(f"{_smoke_id}.tif") as _sf:
            _re = tifffile.imread(_sf)
            assert _re.dtype == np.uint8 and set(np.unique(_re)).issubset({0, 1})
    os.remove(_smoke_tif)
    os.remove(_smoke_zip)

    _smoke_ok = True
    print(f"[SMOKE] PASSED — full chain verified for {_smoke_id}")

except Exception as _se:
    print(f"[SMOKE] FAILED: {_se}")
    import traceback; traceback.print_exc()

try:
    del _smoke_vol, _smoke_logits, _smoke_probs, _smoke_mask
except NameError:
    pass
torch.cuda.empty_cache(); gc.collect()

assert _smoke_ok, "[FATAL] Smoke test failed — do NOT proceed to full inference"
check_budget("Smoke test done")

In [ ]:
# ============================================================
# §10  MAIN INFERENCE LOOP  —  STREAMING ZIP SUBMISSION
# ============================================================
check_budget("Starting inference loop")

# Put all models in eval mode
for m in models:
    m.eval()

red_flags      = []
volume_times   = []
empty_ids      = []
fg_pcts        = []

# np_softmax already defined in smoke test cell above

# Pre-cache volume shapes (metadata only — no pixel data loaded)
_vol_shapes = {}
for _tid in test_ids:
    _tpath = f"{TEST_DIR}/{_tid}.tif"
    try:
        with tifffile.TiffFile(_tpath) as _tf:
            _vol_shapes[_tid] = _tf.series[0].shape
    except Exception:
        print(f"[WARN] Cannot read shape metadata for {_tid}")
print(f"[OK] Pre-cached {len(_vol_shapes)}/{len(test_ids)} volume shapes")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for vi, image_id in enumerate(test_ids):
        t0 = time.time()
        iid = str(image_id)
        fg_pct = 0.0

        # ── Time-aware proactive degradation ──
        vols_left = len(test_ids) - vi
        if vols_left > 0:
            sec_avail = (remaining_h() - 0.5) * 3600  # 0.5 h safety buffer
            sec_per_vol = sec_avail / max(vols_left, 1)
            while sec_per_vol < MAX_SEC_PER_VOL * 0.7 and degradation.level < len(degradation.LEVELS) - 1:
                degradation.degrade()
                sec_per_vol *= 1.4  # each level saves ~30-40 % time

        print(f"\n{'='*60}\n[{vi+1}/{len(test_ids)}] {iid}")

        try:
            # ── Load & normalise ──
            vol_np = load_test_volume(f"{TEST_DIR}/{iid}.tif")
            orig_shape = vol_np.shape  # (D, H, W)

            # ── Degradation-aware config (with model dropping) ──
            cfg = degradation.current()
            tta_mode = cfg["tta"] if TTA_ENABLED else False
            roi = cfg["roi"]
            olap = cfg["overlap"]
            max_m = cfg.get("max_models", len(models))
            vol_models = models[:min(max_m, len(models))]
            vol_weights = active_weights[:len(vol_models)]
            if len(vol_models) < len(models):
                print(f"[DEGRADE] Using {len(vol_models)}/{len(models)} models")

            # ── Ensemble: combine LOGITS across models ──
            weights = vol_weights
            wsum = sum(weights)

            if ENSEMBLE_METHOD == "median" and len(vol_models) > 1:
                all_logits = []
                for mi, mdl in enumerate(vol_models):
                    try:
                        lg = predict_with_tta(mdl, vol_np, roi, olap, NUM_CLASSES, tta_mode)
                    except Exception as oom:
                        if "out of memory" in str(oom).lower() or "oom" in str(oom).lower():
                            print(f"[OOM] model {mi} \u2014 degrading")
                            red_flags.append(f"OOM {iid} model {mi}")
                            torch.cuda.empty_cache()
                            cfg = degradation.degrade()
                            try:
                                lg = predict_with_tta(mdl, vol_np, cfg["roi"], cfg["overlap"],
                                                      NUM_CLASSES, tta_mode=False)
                            except Exception:
                                lg = None
                        else:
                            print(f"[ERROR] model {mi} failed (non-OOM): {oom}")
                            red_flags.append(f"MODEL_FAIL {iid} model {mi}: {oom}")
                            lg = None
                    if lg is not None:
                        all_logits.append(lg)
                if not all_logits:
                    raise RuntimeError(f"All models failed for {iid}")
                ens_logits = np.median(all_logits, axis=0)
                del all_logits
            else:
                ens_logits = None
                used_weight = 0.0
                for mi, mdl in enumerate(vol_models):
                    try:
                        lg = predict_with_tta(mdl, vol_np, roi, olap, NUM_CLASSES, tta_mode)
                    except Exception as oom:
                        if "out of memory" in str(oom).lower() or "oom" in str(oom).lower():
                            print(f"[OOM] model {mi} \u2014 degrading")
                            red_flags.append(f"OOM {iid} model {mi}")
                            torch.cuda.empty_cache()
                            cfg = degradation.degrade()
                            try:
                                lg = predict_with_tta(mdl, vol_np, cfg["roi"], cfg["overlap"],
                                                      NUM_CLASSES, tta_mode=False)
                            except Exception:
                                lg = None
                        else:
                            print(f"[ERROR] model {mi} failed (non-OOM): {oom}")
                            red_flags.append(f"MODEL_FAIL {iid} model {mi}: {oom}")
                            lg = None
                    if lg is not None:
                        w = weights[mi] / wsum
                        ens_logits = lg * w if ens_logits is None else ens_logits + lg * w
                        used_weight += w
                if ens_logits is None:
                    raise RuntimeError(f"All models failed for {iid}")
                if used_weight > 0 and abs(used_weight - 1.0) > 0.01:
                    ens_logits = ens_logits / used_weight  # re-normalise partial weights

            # ── Logits → probability → smoothing → postprocess ──
            probs = np_softmax(ens_logits, axis=-1)
            fg_prob = probs[..., 1]         # surface class
            del probs, ens_logits

            # Gaussian smoothing reduces noise, improves surface continuity
            if PP_SMOOTH_SIGMA > 0:
                fg_prob = ndi.gaussian_filter(fg_prob.astype(np.float64), sigma=PP_SMOOTH_SIGMA).astype(np.float32)

            final_mask = topo_postprocess(
                fg_prob,
                T_low=PP_T_LOW, T_high=PP_T_HIGH,
                z_radius=PP_Z_RADIUS, xy_radius=PP_XY_RADIUS,
                dust_min=PP_DUST_MIN,
            )
            del fg_prob

            # ── Validate ──
            assert final_mask.shape == orig_shape, f"Shape {final_mask.shape} vs {orig_shape}"
            assert final_mask.dtype == np.uint8
            assert set(np.unique(final_mask)).issubset({0, 1})

            fg_pct = 100.0 * final_mask.sum() / final_mask.size
            fg_pcts.append(fg_pct)
            if final_mask.sum() == 0:
                empty_ids.append(iid)
                print(f"[WARN] Empty mask for {iid}")

            # ── Debug viz ──
            if DEBUG_VIZ:
                try:
                    mt = compute_topo_metrics(final_mask); log_topo(mt, iid)
                    if vi == 0:
                        plot_slices(vol_np, final_mask, iid)
                        plot_mip(vol_np, final_mask, iid)
                except Exception as ve:
                    print(f"[WARN] viz: {ve}")

            # ── Write → ZIP → cleanup ──
            out_p = f"{OUTPUT_DIR}/{iid}.tif"
            tifffile.imwrite(out_p, final_mask)
            zf.write(out_p, arcname=f"{iid}.tif")
            os.remove(out_p)

        except Exception as e:
            # ═══════════════════════════════════════════════
            # NEVER SKIP — zero-mask fallback (self-protected)
            # ═══════════════════════════════════════════════
            print(f"[ERROR] {iid}: {e}")
            red_flags.append(f"FALLBACK {iid}: {e}")
            try:
                # Use pre-cached shape (metadata), fall back to re-read
                fb_shape = _vol_shapes.get(iid)
                if fb_shape is None:
                    fb = read_tif_volume(f"{TEST_DIR}/{iid}.tif")
                    fb_shape = fb.shape; del fb
                fb_mask = np.zeros(fb_shape, dtype=np.uint8)
                out_p = f"{OUTPUT_DIR}/{iid}.tif"
                tifffile.imwrite(out_p, fb_mask)
                zf.write(out_p, arcname=f"{iid}.tif")
                os.remove(out_p)
            except Exception as inner_e:
                # Absolute last resort: 1-voxel mask (wrong shape but file exists in ZIP)
                print(f"[CRITICAL] Fallback write also failed for {iid}: {inner_e}")
                red_flags.append(f"CRITICAL_FALLBACK {iid}: {inner_e}")
                fb_mask = np.zeros((1, 1, 1), dtype=np.uint8)
                out_p = f"{OUTPUT_DIR}/{iid}.tif"
                tifffile.imwrite(out_p, fb_mask)
                zf.write(out_p, arcname=f"{iid}.tif")
                try: os.remove(out_p)
                except Exception: pass
            empty_ids.append(iid)

        finally:
            torch.cuda.empty_cache()
            gc.collect()

        dt = time.time() - t0
        volume_times.append(dt)
        print(f"[TIME] {iid}: {dt:.1f}s  FG={fg_pct:.2f}%")
        if dt > MAX_SEC_PER_VOL * 1.5:
            print("[WARN] Slow \u2014 degrading")
            degradation.degrade()

check_budget("Inference complete")

# ============================================================
# §11  FINAL ZIP VALIDATOR  (hard gate)
# ============================================================
print(f"\n{'='*70}\n[VALIDATOR] ZIP VALIDATION\n{'='*70}")

assert os.path.exists(ZIP_PATH), f"[FATAL] ZIP missing: {ZIP_PATH}"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zfiles = [n for n in zf.namelist() if n.endswith(".tif")]

    assert len(zfiles) == len(test_ids), \
        f"[FATAL] {len(zfiles)} files vs {len(test_ids)} IDs"

    nested = [n for n in zfiles if "/" in n]
    assert not nested, f"[FATAL] Nested paths: {nested[:5]}"

    zip_ids = sorted(n.replace(".tif", "") for n in zfiles)
    exp_ids = sorted(str(i) for i in test_ids)
    assert zip_ids == exp_ids, "[FATAL] ID mismatch"

    for fn in [zfiles[0], zfiles[len(zfiles)//2], zfiles[-1]]:
        with zf.open(fn) as f:
            arr = tifffile.imread(f)
            assert arr.dtype == np.uint8, f"{fn} dtype={arr.dtype}"
            assert set(np.unique(arr)).issubset({0, 1}), f"{fn} bad values"

    print(f"[OK] {len(zfiles)} files \u00b7 root-level \u00b7 uint8 {{0,1}} \u00b7 IDs match")

zmb = os.path.getsize(ZIP_PATH) / 1e6
print(f"[OK] {ZIP_PATH} ({zmb:.1f} MB)")

# ============================================================
# §12  RED-FLAG CHECK
# ============================================================
print(f"\n{'='*70}\n[RED-FLAG] Health check\n{'='*70}")
critical = False

if red_flags:
    print(f"[RED-FLAG] {len(red_flags)} issues:")
    for r in red_flags: print(f"   \u2022 {r}")
    critical = True

if empty_ids:
    print(f"[RED-FLAG] {len(empty_ids)} empty masks: {empty_ids[:10]}")
    critical = True

if volume_times:
    avg_t = np.mean(volume_times)
    print(f"[STATS] Time: avg={avg_t:.1f}s  max={np.max(volume_times):.1f}s  "
          f"total={sum(volume_times)/60:.1f} min")
    if len(volume_times) > 5:
        f5, l5 = np.mean(volume_times[:5]), np.mean(volume_times[-5:])
        if l5 > f5 * 1.5:
            print(f"[RED-FLAG] RAM creep: first5={f5:.1f}s \u2192 last5={l5:.1f}s")
            critical = True

if fg_pcts:
    print(f"[STATS] FG: avg={np.mean(fg_pcts):.2f}%  "
          f"min={np.min(fg_pcts):.2f}%  max={np.max(fg_pcts):.2f}%")

h = elapsed_h()
print(f"\n{'='*70}")
if critical:
    print(f"[WARN] SUBMISSION READY WITH WARNINGS ({h:.2f} h)")
else:
    print(f"[OK] SUBMISSION READY \u2014 ALL VALIDATORS PASSED ({h:.2f} h)")